# Subreddit groups v7 (v4 sample, pre-accusation subreddits)

Builds on `subreddit_groups_v6_test_group.ipynb` with the 2026-10-07 decisions:
- new **`entertainment`** group = gaming + sports + TV/film/anime/music (and fan communities/memes about them). Every v6
  `other` subreddit was re-read by Claude agents per `subreddit_entertainment_guide.md` and labelled entertainment or
  other (`subreddit_review_output/subreddit_llm_labels_entertainment_v7.csv`);
- the **`test` group is folded back into `other`**, kept as the flag `test_subreddit`;
- mental health, kids'-content spam, tech, hobbies, identity, etc. stay in `other` (too few distinct accounts, or not chosen).

Groups: profile, adult, crypto, politics, karma_farming, general, news, sales, location, entertainment, other.
Flags: `later_banned`, `private`, `test_subreddit`. Outputs (never overwritten): `subreddit_groups_v7.parquet` + review xlsx.

In [ ]:
# Libraries, folders, and the group list (adds entertainment; test is no longer a group).
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

SAMPLE_DIR = 'sample_1000_v4_output'
OUT_DIR = 'subreddit_review_output'
GROUPS = ['profile', 'adult', 'crypto', 'politics', 'karma_farming', 'general', 'news', 'sales', 'location', 'entertainment', 'other']

# Write a file only if it does not exist yet (never overwrite); .xlsx goes to Excel, anything else to parquet.
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    if name.endswith('.xlsx'):
        df.to_excel(path, index=False, freeze_panes=(1, 1))
    else:
        df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

In [ ]:
# Start from v6 and the agent labels that split the v6 other group into entertainment and other.
g = pd.read_parquet(os.path.join(OUT_DIR, 'subreddit_groups_v6.parquet')).rename(columns={'group_v5': 'group_v5_'})
ent = pd.read_csv(os.path.join(OUT_DIR, 'subreddit_llm_labels_entertainment_v7.csv'))
assert ent['subreddit'].is_unique and set(ent['ent']) <= {'entertainment', 'other'}
# Test is kept as a flag; those subreddits move to other.
g['group_v6'] = g['group']
g['test_subreddit'] = g['group'].eq('test')
g.loc[g['test_subreddit'], 'group'] = 'other'
v6_other = g['group_v6'].eq('other')
# The entertainment labels must cover exactly the v6 other group.
assert set(ent['subreddit']) == set(g.loc[v6_other, 'subreddit']), 'entertainment labels must cover exactly the v6 other group'
lab = ent.set_index('subreddit')
# Move labelled subreddits to entertainment and keep the agent's confidence and reason.
is_ent = v6_other & g['subreddit'].map(lab['ent']).eq('entertainment')
g.loc[is_ent, 'group'] = 'entertainment'
g.loc[is_ent, 'group_source'] = 'agent_entertainment'
g['entertainment_confidence'] = g['subreddit'].map(lab['confidence'])
g['entertainment_reason'] = g['subreddit'].map(lab['reason'])
g = g.drop(columns='group_v5_')
assert set(g['group']) <= set(GROUPS)
# How v6 other and test subreddits were split.
print(pd.crosstab(g['group_v6'], g['group']).loc[['other', 'test'], ['entertainment', 'other']].to_string())

In [ ]:
# Every post and comment the sample made before its first (edit-adjusted) accusation, tagged with its subreddit's group.
accounts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_accounts.parquet')).set_index('username_key')
acts = []
for f in ['sample_posts.parquet', 'sample_comments.parquet']:
    d = pd.read_parquet(os.path.join(SAMPLE_DIR, f), columns=['outer_key', 'created_utc', 'subreddit'])
    acts.append(d[d['created_utc'] < d['outer_key'].map(accounts['first_mention_adj_utc'])])
acts = pd.concat(acts)
acts['group'] = acts['subreddit'].map(g.set_index('subreddit')['group'])
assert acts['group'].notna().all()
# Share of a group's actions made by its single busiest account (high = one account dominates).
def top_share(x):
    per = x.groupby('outer_key').size()
    return per.max() / per.sum()
# Per group: subreddits, actions and accounts, and the share of all pre-accusation actions.
summary = pd.DataFrame({'subreddits': g['group'].value_counts(), 'actions': acts['group'].value_counts(),
                        'accounts': acts.groupby('group')['outer_key'].nunique(),
                        'top_account_share': acts.groupby('group').apply(top_share).round(2)}).reindex(GROUPS)
summary['pct_actions'] = (100 * summary['actions'] / summary['actions'].sum()).round(1)
summary

In [ ]:
# Save the groups, and a review sheet sorted by group and use.
save_new(g, 'subreddit_groups_v7.parquet')
review = g.sort_values(['group', 'actions'], ascending=[True, False])[
    ['subreddit', 'group', 'group_v6', 'group_source', 'confidence', 'reason', 'entertainment_confidence', 'entertainment_reason',
     'later_banned', 'private', 'test_subreddit', 'archived_nsfw_share', 'actions', 'accounts', 'title', 'public_description']]
save_new(review, 'subreddit_groups_v7_review.xlsx')